# Week 4: Baseline Model
- Train a linear regression as the first model
- Evaluate using R^2 on test set
- Record baseline results

In [1]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from pathlib import Path

data_clean = pd.read_csv("data_clean.csv")

In [2]:
# loading in test/train split data
filepath = r"C:\Users\julia\downloads\IDX_summer_internship"

train_df = pd.read_csv(filepath + r"\train_cleaned.csv", low_memory=False)
test_df = pd.read_csv(filepath + r"\test_cleaned.csv", low_memory=False)

print(f"Training rows: {len(train_df):,}")
print(f"Testing rows:  {len(test_df):,}")
print(f"Columns: {train_df.shape[1]}")

Training rows: 118,196
Testing rows:  12,024
Columns: 299


In [20]:
# chosing target variable, creating x and y

target = "ClosePrice"
x_train = train_df.drop(columns=["ClosePrice"]) # we want to train on ClosePrice so we remove it here
x_test = test_df.drop(columns=["ClosePrice"])

y_train = train_df['ClosePrice']
y_test = test_df['ClosePrice']

drop_cols = [                   # these are based on what Aidan said in the slack
    "CloseDate",                # stuff like days on market are only known after the close
    "ListPrice",                # everything else introduces leakages
    "OriginalListPrice",
    "ListingKey",
    "ListingKeyNumeric",
    "DaysOnMarket"
]

x_train = x_train.drop(columns=drop_cols, errors="ignore") # dropping the rest of the leakage or unnecessary columns
x_test = x_test.drop(columns=drop_cols, errors="ignore")

drop_cols.extend([             # string values that interrupt the linear regression that made it through preprocessing
    "City",
    "HighSchoolDistrict",
    "PostalCode"
])

x_train = x_train.drop(columns=drop_cols, errors="ignore")
x_test = x_test.drop(columns=drop_cols, errors="ignore")

In [25]:
# training the linear regression model

from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(x_train, y_train)

LinearRegression()

In [27]:
# predict on the test set
y_pred = lr.predict(x_test)

In [29]:
# evaluate on R^2

from sklearn.metrics import r2_score, mean_absolute_error

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Baseline R^2: {r2:.4f}")
print(f"Baseline MAE: ${mae:,.2f}")

Baseline R^2: 0.0390
Baseline MAE: $696,625.65


In [33]:
# record baseline
baseline_results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "R2": [r2],
    "MAE": [mae]
})

baseline_results

,Model,R2,MAE
0,Linear Regression,0.039002,696625.652372


In [44]:
coef_df = pd.DataFrame({
    "Feature": x_train.columns,
    "Coefficient": lr.coef_
})

coef_df = coef_df.reindex(
    coef_df["Coefficient"].abs().sort_values(ascending=False).index
)

coef_df.head(50)

,Feature,Coefficient
231,BuyerAgentAOR_TUOLUMNECOUNTYASSOCIATIONOFREALTORS,-1.334854e+07
268,ListAgentAOR_PACIFICSOUTHWEST,4.759134e+06
14,LotSizeArea,3.694654e+06
262,ListAgentAOR_NORTHSANDIEGOCOUNTY,3.536636e+06
4,LivingArea,3.457045e+06
76,CountyOrParish_TRINITY,-2.790666e+06
212,BuyerAgentAOR_PACIFICSOUTHWEST,-2.641571e+06
223,BuyerAgentAOR_SANLUISOBISPOCOASTAL,2.459454e+06
29,CountyOrParish_FOREIGN COUNTRY,-2.447643e+06
140,BuyerOfficeAOR_SOUTHWESTLOUISIANAASSOCIATIONOF...,-2.363629e+06
